# Klasifikasi Citra Teks Menggunakan Classifier + Fitur GLCM dan Arsitektur Deep Learning 

Notebook ini membandingkan dua algoritma klasifikasi berbasis fitur tekstur **Gray-Level Co-occurrence Matrix (GLCM)**:

1. **Gaussian Naive Bayes + GLCM** — 5 konfigurasi eksperimen.
2. **Random Forest + GLCM** — 5 konfigurasi eksperimen.

Setiap eksperimen menggunakan konfigurasi hyperparameter yang berbeda. Kedua algoritma menggunakan fitur GLCM dan pembagian data train/test yang sama agar perbandingan lebih konsisten.

## Struktur folder dataset

Dataset berada pada folder `images` yang bersebelahan dengan notebook. Setiap kelas memiliki subfolder sendiri di dalam `images`, dan nama subfolder akan digunakan sebagai label kelas.

```text
images/
├── kelas_1/
│   ├── gambar_001.png
│   └── gambar_002.jpg
├── kelas_2/
│   ├── gambar_001.png
│   └── gambar_002.jpg
└── kelas_3/
    ├── gambar_001.png
    └── gambar_002.jpg
```

Notebook membaca seluruh subfolder kelas di dalam `images` secara rekursif. File gambar yang rusak atau tidak dapat dibaca akan dilewati.

**Catatan evaluasi:** metrik dihitung pada data test. Precision, Recall, dan F1-Score secara default menggunakan `weighted average`, sehingga memperhitungkan jumlah sampel pada setiap kelas. Waktu pelatihan dihitung dalam detik dan mencakup proses `fit` model (termasuk `StandardScaler` untuk Naive Bayes), tetapi tidak mencakup pembacaan gambar maupun ekstraksi GLCM.

In [2]:
# Jika library belum tersedia, jalankan di terminal:
# pip install -r requirements.txt

from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError
from skimage.feature import graycomatrix, graycoprops

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
from IPython.display import display

warnings.filterwarnings("ignore", category=RuntimeWarning)

print("Semua library berhasil diimpor.")

Semua library berhasil diimpor.


## 1. Konfigurasi awal

`DATA_DIR` diarahkan ke folder `images` (relatif terhadap notebook). Setiap subfolder langsung di dalamnya diperlakukan sebagai satu kelas. Parameter lain dapat disesuaikan, tetapi sebaiknya gunakan konfigurasi yang sama saat membandingkan kedua algoritma.

In [3]:
# Folder dataset: setiap subfolder di dalam "images" mewakili satu kelas.
DATA_DIR = Path("images").resolve()

# Ukuran gambar setelah diubah menjadi grayscale (tinggi, lebar)
IMAGE_SIZE = (128, 128)

# Pembagian data
TEST_SIZE = 0.20
RANDOM_STATE = 42

# Untuk klasifikasi multikelas/imbalanced, weighted biasanya sesuai sebagai nilai awal.
# Pilihan lain: "macro" atau "micro".
AVERAGE_METHOD = "weighted"

# Ekstensi gambar yang akan dibaca
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

# Konfigurasi GLCM yang dipakai sama untuk kedua algoritma dan seluruh eksperimen.
GLCM_CONFIG = {
    "levels": 32,
    "distances": [1, 2],
    "angles_degrees": [0, 45, 90, 135],
    "symmetric": True,
    "normed": True,
}

# Fitur GLCM yang dihitung. Setiap properti diringkas menjadi mean dan standard deviation.
GLCM_PROPERTIES = [
    "contrast",
    "dissimilarity",
    "homogeneity",
    "energy",
    "correlation",
    "ASM",
]

print("DATA_DIR:", DATA_DIR.resolve())
print("Konfigurasi GLCM:", GLCM_CONFIG)

DATA_DIR: C:\Users\Valentino\OneDrive - Universitas Tarumanagara\Documents\Kuliah\Semester 7\Machine Learning\Tugas\Tugas Klasifikasi Data Citra\Project\images
Konfigurasi GLCM: {'levels': 32, 'distances': [1, 2], 'angles_degrees': [0, 45, 90, 135], 'symmetric': True, 'normed': True}


## 2. Membaca dataset gambar

Label kelas diambil dari nama subfolder langsung di dalam `images`. Gambar diubah ke grayscale dan di-resize ke ukuran yang ditentukan pada konfigurasi.

In [4]:
def load_image_dataset(data_dir, image_size=(128, 128), extensions=None):
    """Membaca citra dari struktur folder: images/nama_kelas/file_gambar."""
    data_dir = Path(data_dir)
    extensions = extensions or SUPPORTED_EXTENSIONS

    if not data_dir.exists() or not data_dir.is_dir():
        raise FileNotFoundError(
            f"Folder dataset tidak ditemukan: {data_dir}. Silakan periksa DATA_DIR."
        )

    class_dirs = sorted(
        [p for p in data_dir.iterdir() if p.is_dir() and not p.name.startswith(".")]
    )
    if len(class_dirs) < 2:
        raise ValueError(
            "Dataset harus memiliki minimal dua subfolder kelas di dalam DATA_DIR."
        )

    images = []
    labels = []
    skipped_files = []

    # PIL menerima ukuran dalam format (lebar, tinggi).
    resize_to = (int(image_size[1]), int(image_size[0]))

    for class_dir in class_dirs:
        image_paths = sorted(
            p for p in class_dir.rglob("*")
            if p.is_file() and p.suffix.lower() in extensions
        )

        for image_path in image_paths:
            try:
                with Image.open(image_path) as image:
                    gray_image = image.convert("L").resize(
                        resize_to, Image.Resampling.LANCZOS
                    )
                    images.append(np.asarray(gray_image, dtype=np.uint8))
                    labels.append(class_dir.name)
            except (UnidentifiedImageError, OSError, ValueError) as error:
                skipped_files.append((str(image_path), str(error)))

    if not images:
        raise ValueError(
            "Tidak ada gambar yang berhasil dibaca. Periksa ekstensi dan isi folder dataset."
        )

    X_images = np.stack(images)
    y_labels = np.asarray(labels)

    print(f"Jumlah gambar berhasil dibaca : {len(X_images)}")
    print(f"Jumlah kelas                 : {len(np.unique(y_labels))}")
    print(f"Jumlah file dilewati         : {len(skipped_files)}")

    if skipped_files:
        print("Contoh file yang dilewati:")
        for path, error in skipped_files[:5]:
            print(f"- {path} ({error})")

    return X_images, y_labels


images, labels = load_image_dataset(
    DATA_DIR,
    image_size=IMAGE_SIZE,
    extensions=SUPPORTED_EXTENSIONS,
)

class_distribution = (
    pd.Series(labels, name="Kelas")
    .value_counts()
    .sort_index()
    .rename("Jumlah Gambar")
    .to_frame()
)
display(class_distribution)

Jumlah gambar berhasil dibaca : 2311
Jumlah kelas                 : 44
Jumlah file dilewati         : 0


,Jumlah Gambar
Kelas,
E210077,47
E230002,56
E230005,50
E230011,52
E230017,50
E230018,51
E230020,50
E230028,50
E230031,51


## 3. Membagi data train dan test

Pembagian dilakukan satu kali dengan `stratify` agar proporsi kelas pada train dan test tetap terjaga. Indeks yang sama digunakan untuk semua eksperimen Naive Bayes dan Random Forest.

In [5]:
class_counts = pd.Series(labels).value_counts()

if (class_counts < 2).any():
    rare_classes = class_counts[class_counts < 2].index.tolist()
    raise ValueError(
        "Setiap kelas memerlukan minimal 2 gambar agar stratified split dapat dilakukan. "
        f"Kelas dengan sampel kurang dari 2: {rare_classes}"
    )

all_indices = np.arange(len(labels))
try:
    train_indices, test_indices = train_test_split(
        all_indices,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=labels,
    )
except ValueError as error:
    raise ValueError(
        "Pembagian train/test gagal. Pastikan setiap kelas memiliki cukup gambar; "
        "bila dataset kecil, perbesar jumlah gambar atau sesuaikan TEST_SIZE. "
        f"Detail: {error}"
    ) from error

print(f"Jumlah data train: {len(train_indices)}")
print(f"Jumlah data test : {len(test_indices)}")
print("\nDistribusi kelas pada data train:")
display(pd.Series(labels[train_indices]).value_counts().sort_index().to_frame("Train"))
print("Distribusi kelas pada data test:")
display(pd.Series(labels[test_indices]).value_counts().sort_index().to_frame("Test"))

Jumlah data train: 1848
Jumlah data test : 463

Distribusi kelas pada data train:


,Train
E210077,38
E230002,45
E230005,40
E230011,42
E230017,40
E230018,41
E230020,40
E230028,40
E230031,41
E230034,40


Distribusi kelas pada data test:


,Test
E210077,9
E230002,11
E230005,10
E230011,10
E230017,10
E230018,10
E230020,10
E230028,10
E230031,10
E230034,10


## 4. Ekstraksi fitur GLCM

Setiap gambar grayscale dikuantisasi ke sejumlah `levels` agar dapat diproses oleh GLCM. Dari setiap properti GLCM dihitung **mean** dan **standard deviation** pada seluruh kombinasi jarak dan sudut. Dengan demikian, setiap gambar menghasilkan 12 fitur (6 properti × 2 statistik).

Properti yang digunakan adalah contrast, dissimilarity, homogeneity, energy, correlation, dan ASM.

In [6]:
def extract_glcm_features(image_array, glcm_config, properties=None):
    """Menghasilkan vektor fitur GLCM untuk satu citra grayscale uint8."""
    properties = properties or GLCM_PROPERTIES
    levels = int(glcm_config["levels"])

    if levels < 2 or levels > 256:
        raise ValueError("GLCM_CONFIG['levels'] harus berada pada rentang 2 sampai 256.")

    # Kuantisasi nilai piksel 0..255 menjadi 0..levels-1.
    quantized = (
        image_array.astype(np.uint16) * levels // 256
    ).astype(np.uint8)
    quantized = np.clip(quantized, 0, levels - 1)

    angles = np.deg2rad(glcm_config["angles_degrees"])
    glcm = graycomatrix(
        quantized,
        distances=glcm_config["distances"],
        angles=angles,
        levels=levels,
        symmetric=glcm_config["symmetric"],
        normed=glcm_config["normed"],
    )

    feature_values = []
    for prop in properties:
        values = np.asarray(graycoprops(glcm, prop), dtype=np.float64)
        values = np.nan_to_num(values, nan=0.0, posinf=0.0, neginf=0.0)
        feature_values.extend([float(np.mean(values)), float(np.std(values))])

    return np.asarray(feature_values, dtype=np.float64)


def build_glcm_dataset(image_batch, glcm_config):
    """Mengekstrak fitur GLCM untuk seluruh gambar."""
    feature_matrix = np.vstack([
        extract_glcm_features(image, glcm_config)
        for image in image_batch
    ])
    return feature_matrix


feature_start = time.perf_counter()
X_features = build_glcm_dataset(images, GLCM_CONFIG)
feature_extraction_seconds = time.perf_counter() - feature_start

feature_names = [
    f"{prop}_{stat}"
    for prop in GLCM_PROPERTIES
    for stat in ("mean", "std")
]

print("Ukuran matriks fitur:", X_features.shape)
print(f"Waktu ekstraksi seluruh fitur GLCM: {feature_extraction_seconds:.4f} detik")
display(pd.DataFrame(X_features, columns=feature_names).head())

Ukuran matriks fitur: (2311, 12)
Waktu ekstraksi seluruh fitur GLCM: 2.7788 detik


,contrast_mean,contrast_std,dissimilarity_mean,dissimilarity_std,homogeneity_mean,homogeneity_std,energy_mean,energy_std,correlation_mean,correlation_std,ASM_mean,ASM_std
0,7.548804,1.981492,1.089389,0.243406,0.786194,0.033695,0.693239,0.015111,0.619617,0.098755,0.480809,0.021140
1,10.128336,2.865877,1.235370,0.294628,0.789941,0.031974,0.726077,0.015703,0.598577,0.113150,0.527434,0.022935
2,11.727509,2.884189,1.653077,0.339038,0.696193,0.041807,0.614741,0.019238,0.571806,0.104852,0.378276,0.024032
3,12.556471,3.095238,1.680467,0.337609,0.703431,0.037769,0.626597,0.018466,0.587331,0.101325,0.392965,0.023405
4,7.869761,2.164908,0.962190,0.215183,0.836871,0.022822,0.790358,0.010549,0.666064,0.091535,0.624777,0.016736


## 5. Konfigurasi lima eksperimen

GLCM dibuat tetap untuk seluruh eksperimen agar perbedaan hasil terutama berasal dari konfigurasi classifier. Untuk Gaussian Naive Bayes, parameter yang divariasikan adalah `var_smoothing` dan pendekatan prior kelas. Untuk Random Forest, beberapa parameter pohon dan pembobotan kelas divariasikan.

In [7]:
# Lima konfigurasi Gaussian Naive Bayes.
# prior_mode="empirical": prior kelas dipelajari dari proporsi data train.
# prior_mode="uniform"  : semua kelas diberi prior yang sama.
NAIVE_BAYES_EXPERIMENTS = [
    {"var_smoothing": 1e-9, "prior_mode": "empirical"},
    {"var_smoothing": 1e-8, "prior_mode": "uniform"},
    {"var_smoothing": 1e-7, "prior_mode": "empirical"},
    {"var_smoothing": 1e-6, "prior_mode": "uniform"},
    {"var_smoothing": 1e-5, "prior_mode": "empirical"},
]

# Lima konfigurasi Random Forest.
RANDOM_FOREST_EXPERIMENTS = [
    {
        "n_estimators": 100,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": "sqrt",
        "class_weight": None,
    },
    {
        "n_estimators": 200,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": "sqrt",
        "class_weight": "balanced",
    },
    {
        "n_estimators": 300,
        "max_depth": 20,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": "log2",
        "class_weight": None,
    },
    {
        "n_estimators": 200,
        "max_depth": 10,
        "min_samples_split": 5,
        "min_samples_leaf": 2,
        "max_features": 0.8,
        "class_weight": "balanced_subsample",
    },
    {
        "n_estimators": 400,
        "max_depth": None,
        "min_samples_split": 10,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
        "class_weight": None,
    },
]

print("Konfigurasi Naive Bayes:")
nb_config_table = pd.DataFrame(NAIVE_BAYES_EXPERIMENTS)
nb_config_table.insert(0, "Eksperimen ke", range(1, len(nb_config_table) + 1))
display(nb_config_table)

print("Konfigurasi Random Forest:")
rf_config_table = pd.DataFrame(RANDOM_FOREST_EXPERIMENTS)
rf_config_table.insert(0, "Eksperimen ke", range(1, len(rf_config_table) + 1))
display(rf_config_table)

Konfigurasi Naive Bayes:


,Eksperimen ke,var_smoothing,prior_mode
0,1,1.000000e-09,empirical
1,2,1.000000e-08,uniform
2,3,1.000000e-07,empirical
3,4,1.000000e-06,uniform
4,5,1.000000e-05,empirical


Konfigurasi Random Forest:


,Eksperimen ke,n_estimators,max_depth,min_samples_split,min_samples_leaf,max_features,class_weight
0,1,100,NaN,2,1,sqrt,NaN
1,2,200,NaN,2,1,sqrt,balanced
2,3,300,20.0,2,1,log2,NaN
3,4,200,10.0,5,2,0.8,balanced_subsample
4,5,400,NaN,10,2,sqrt,NaN


## 6. Fungsi evaluasi eksperimen

Setiap konfigurasi dilatih menggunakan data train yang sama, kemudian dievaluasi pada data test yang sama. Waktu pelatihan hanya mengukur pemanggilan `fit`.

In [8]:
def run_classifier_experiments(
    algorithm,
    configurations,
    X,
    y,
    train_idx,
    test_idx,
    average_method="weighted",
    random_state=42,
):
    """Menjalankan lima eksperimen dan mengembalikan tabel metrik serta model."""
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    metric_rows = []
    fitted_models = []

    for experiment_number, config in enumerate(configurations, start=1):
        if algorithm == "naive_bayes":
            if config["prior_mode"] == "uniform":
                class_count = len(np.unique(y_train))
                class_priors = np.full(class_count, 1.0 / class_count)
            elif config["prior_mode"] == "empirical":
                class_priors = None
            else:
                raise ValueError(f"prior_mode tidak dikenal: {config['prior_mode']}")

            estimator = Pipeline([
                ("scaler", StandardScaler()),
                ("classifier", GaussianNB(
                    var_smoothing=config["var_smoothing"],
                    priors=class_priors,
                )),
            ])

        elif algorithm == "random_forest":
            estimator = RandomForestClassifier(
                n_estimators=config["n_estimators"],
                max_depth=config["max_depth"],
                min_samples_split=config["min_samples_split"],
                min_samples_leaf=config["min_samples_leaf"],
                max_features=config["max_features"],
                class_weight=config["class_weight"],
                random_state=random_state,
                n_jobs=-1,
            )
        else:
            raise ValueError("algorithm harus 'naive_bayes' atau 'random_forest'.")

        start_time = time.perf_counter()
        estimator.fit(X_train, y_train)
        training_seconds = time.perf_counter() - start_time

        predictions = estimator.predict(X_test)

        metric_rows.append({
            "Eksperimen ke": experiment_number,
            "Precision": precision_score(
                y_test, predictions, average=average_method, zero_division=0
            ),
            "Recall": recall_score(
                y_test, predictions, average=average_method, zero_division=0
            ),
            "Akurasi": accuracy_score(y_test, predictions),
            "F1-Score": f1_score(
                y_test, predictions, average=average_method, zero_division=0
            ),
            "Waktu Pelatihan": training_seconds,
        })

        fitted_models.append({
            "experiment": experiment_number,
            "config": config.copy(),
            "model": estimator,
            "predictions": predictions,
        })

        print(
            f"Eksperimen {experiment_number}/5 selesai | "
            f"konfigurasi={config} | waktu fit={training_seconds:.4f} detik"
        )

    results_df = pd.DataFrame(metric_rows)
    metric_columns = [
        "Precision", "Recall", "Akurasi", "F1-Score", "Waktu Pelatihan"
    ]
    average_values = results_df[metric_columns].mean()
    average_row = {"Eksperimen ke": "Rata-rata", **average_values.to_dict()}
    result_table = pd.concat(
        [results_df, pd.DataFrame([average_row])],
        ignore_index=True,
    )

    print("\nTabel hasil eksperimen:")
    display(result_table.round(4))

    return results_df, result_table, fitted_models


# Indeks train/test yang sama dipakai oleh kedua algoritma.
X_train = X_features[train_indices]
X_test = X_features[test_indices]
y_train = labels[train_indices]
y_test = labels[test_indices]

print(f"Fitur train: {X_train.shape}; fitur test: {X_test.shape}")

Fitur train: (1848, 12); fitur test: (463, 12)


## 7. Eksperimen 1 — Gaussian Naive Bayes + GLCM

Gaussian Naive Bayes digunakan karena fitur GLCM berupa nilai numerik kontinu. `StandardScaler` diterapkan di dalam pipeline dan dipelajari hanya dari data train.

In [9]:
nb_results, nb_result_table, nb_models = run_classifier_experiments(
    algorithm="naive_bayes",
    configurations=NAIVE_BAYES_EXPERIMENTS,
    X=X_features,
    y=labels,
    train_idx=train_indices,
    test_idx=test_indices,
    average_method=AVERAGE_METHOD,
    random_state=RANDOM_STATE,
)

Eksperimen 1/5 selesai | konfigurasi={'var_smoothing': 1e-09, 'prior_mode': 'empirical'} | waktu fit=0.0095 detik
Eksperimen 2/5 selesai | konfigurasi={'var_smoothing': 1e-08, 'prior_mode': 'uniform'} | waktu fit=0.0104 detik
Eksperimen 3/5 selesai | konfigurasi={'var_smoothing': 1e-07, 'prior_mode': 'empirical'} | waktu fit=0.0114 detik
Eksperimen 4/5 selesai | konfigurasi={'var_smoothing': 1e-06, 'prior_mode': 'uniform'} | waktu fit=0.0081 detik
Eksperimen 5/5 selesai | konfigurasi={'var_smoothing': 1e-05, 'prior_mode': 'empirical'} | waktu fit=0.0084 detik

Tabel hasil eksperimen:


,Eksperimen ke,Precision,Recall,Akurasi,F1-Score,Waktu Pelatihan
0,1,0.4408,0.3952,0.3952,0.3750,0.0095
1,2,0.4376,0.3952,0.3952,0.3737,0.0104
2,3,0.4408,0.3952,0.3952,0.3750,0.0114
3,4,0.4376,0.3952,0.3952,0.3737,0.0081
4,5,0.4408,0.3952,0.3952,0.3750,0.0084
5,Rata-rata,0.4395,0.3952,0.3952,0.3745,0.0096


## 8. Eksperimen 2 — Random Forest + GLCM

Random Forest tidak memerlukan standardisasi fitur. `n_jobs=-1` digunakan agar proses pelatihan dapat memanfaatkan beberapa core CPU.

In [10]:
rf_results, rf_result_table, rf_models = run_classifier_experiments(
    algorithm="random_forest",
    configurations=RANDOM_FOREST_EXPERIMENTS,
    X=X_features,
    y=labels,
    train_idx=train_indices,
    test_idx=test_indices,
    average_method=AVERAGE_METHOD,
    random_state=RANDOM_STATE,
)

Eksperimen 1/5 selesai | konfigurasi={'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'class_weight': None} | waktu fit=0.2637 detik
Eksperimen 2/5 selesai | konfigurasi={'n_estimators': 200, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'class_weight': 'balanced'} | waktu fit=0.5297 detik
Eksperimen 3/5 selesai | konfigurasi={'n_estimators': 300, 'max_depth': 20, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'class_weight': None} | waktu fit=0.7476 detik
Eksperimen 4/5 selesai | konfigurasi={'n_estimators': 200, 'max_depth': 10, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 0.8, 'class_weight': 'balanced_subsample'} | waktu fit=0.9440 detik
Eksperimen 5/5 selesai | konfigurasi={'n_estimators': 400, 'max_depth': None, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'class_weight': None} | waktu fit=0.9023 detik

Tab

,Eksperimen ke,Precision,Recall,Akurasi,F1-Score,Waktu Pelatihan
0,1,0.5214,0.5205,0.5205,0.5108,0.2637
1,2,0.5494,0.5486,0.5486,0.5406,0.5297
2,3,0.5331,0.5313,0.5313,0.5214,0.7476
3,4,0.5050,0.4795,0.4795,0.4630,0.9440
4,5,0.5166,0.5097,0.5097,0.4995,0.9023
5,Rata-rata,0.5251,0.5179,0.5179,0.5071,0.6775


## 9. Ringkasan perbandingan model terbaik

Model terbaik dari setiap algoritma dipilih berdasarkan F1-Score pada data test. Bagian ini membantu melihat konfigurasi terbaik dari kedua pendekatan.

In [11]:
best_nb_index = int(nb_results["F1-Score"].idxmax())
best_rf_index = int(rf_results["F1-Score"].idxmax())

best_nb = nb_models[best_nb_index]
best_rf = rf_models[best_rf_index]

best_model_comparison = pd.DataFrame([
    {
        "Algoritma": "Gaussian Naive Bayes + GLCM",
        "Eksperimen terbaik": best_nb["experiment"],
        "Precision": nb_results.loc[best_nb_index, "Precision"],
        "Recall": nb_results.loc[best_nb_index, "Recall"],
        "Akurasi": nb_results.loc[best_nb_index, "Akurasi"],
        "F1-Score": nb_results.loc[best_nb_index, "F1-Score"],
        "Waktu Pelatihan": nb_results.loc[best_nb_index, "Waktu Pelatihan"],
        "Konfigurasi": str(best_nb["config"]),
    },
    {
        "Algoritma": "Random Forest + GLCM",
        "Eksperimen terbaik": best_rf["experiment"],
        "Precision": rf_results.loc[best_rf_index, "Precision"],
        "Recall": rf_results.loc[best_rf_index, "Recall"],
        "Akurasi": rf_results.loc[best_rf_index, "Akurasi"],
        "F1-Score": rf_results.loc[best_rf_index, "F1-Score"],
        "Waktu Pelatihan": rf_results.loc[best_rf_index, "Waktu Pelatihan"],
        "Konfigurasi": str(best_rf["config"]),
    },
])

display(best_model_comparison.round(4))

,Algoritma,Eksperimen terbaik,Precision,Recall,Akurasi,F1-Score,Waktu Pelatihan,Konfigurasi
0,Gaussian Naive Bayes + GLCM,1,0.4408,0.3952,0.3952,0.3750,0.0095,"{'var_smoothing': 1e-09, 'prior_mode': 'empiri..."
1,Random Forest + GLCM,2,0.5494,0.5486,0.5486,0.5406,0.5297,"{'n_estimators': 200, 'max_depth': None, 'min_..."


## 10. Menyimpan hasil eksperimen (opsional)

Jalankan sel berikut untuk menyimpan tabel hasil ke file CSV pada working directory notebook.

In [12]:
SAVE_RESULTS = True

if SAVE_RESULTS:
    nb_result_table.to_csv("hasil_naive_bayes_glcm.csv", index=False)
    rf_result_table.to_csv("hasil_random_forest_glcm.csv", index=False)
    best_model_comparison.to_csv("perbandingan_model_terbaik_glcm.csv", index=False)
    print("File hasil berhasil disimpan:")
    print("- hasil_naive_bayes_glcm.csv")
    print("- hasil_random_forest_glcm.csv")
    print("- perbandingan_model_terbaik_glcm.csv")
else:
    print("Penyimpanan hasil dinonaktifkan.")

File hasil berhasil disimpan:
- hasil_naive_bayes_glcm.csv
- hasil_random_forest_glcm.csv
- perbandingan_model_terbaik_glcm.csv


## Catatan interpretasi

- **Precision, Recall, F1-Score:** menggunakan `AVERAGE_METHOD` (default `weighted`). Nilai yang ditampilkan berada pada rentang 0–1; semakin tinggi umumnya semakin baik.
- **Akurasi:** proporsi prediksi benar dari seluruh data test.
- **Waktu Pelatihan:** durasi pemanggilan `fit` dalam detik. Waktu ini tidak memasukkan pemuatan data dan ekstraksi GLCM.
- **Rata-rata:** rata-rata aritmetika hasil dari lima konfigurasi eksperimen, bukan hasil model ensemble.
- Hasil eksperimen bergantung pada dataset, pembagian train/test, ukuran gambar, dan konfigurasi GLCM. Untuk laporan penelitian, cantumkan seluruh konfigurasi dan `RANDOM_STATE` yang digunakan.